# Notebook 1: C-Trapezoid Method (Paper's Formulas)
Computes H, G, K using **only 7 octile values** via C-Trapezoid and Midpoint approximations.
H and G match the paper's Tables 5--6. **K (C-Trapezoid)** uses the corrected Table~7 formula \((3a-3b+c)/\mathrm{den}\) so it agrees with \(K=(H_L+H_R)/(2H)\) (the printed \(6a-6b+2c\) form was off by a factor of two).

In [ ]:
!pip install -q transformers torch safetensors pandas matplotlib seaborn

In [ ]:
import re, os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
OUT = 'outputs_ctrap'
os.makedirs(OUT, exist_ok=True)
print('Ready.')

In [ ]:
# ==============================================
# CORE FORMULAS — from paper Tables 5, 6, 7
# ==============================================

def compute_octiles(w):
    return np.quantile(w, [1/8, 2/8, 3/8, 4/8, 5/8, 6/8, 7/8], method='linear')

# --- C-TRAPEZOID (paper's best approximation) ---

def H_ctrap(O1,O2,O3,O4,O5,O6,O7):
    """Paper Table 5: C-Trapezoid MAD.  H = IR - IL."""
    Q0 = 4*O1 - 6*O2 + 4*O3 - O4
    Q1 = 4*O7 - 6*O6 + 4*O5 - O4
    h = 1/8
    IL = h*(Q0+O1)/2 + h*(O1+O2)/2 + h*(O2+O3)/2 + h*(O3+O4)/2
    IR = h*(O4+O5)/2 + h*(O5+O6)/2 + h*(O6+O7)/2 + h*(O7+Q1)/2
    return IR - IL

def G_ctrap(O1,O2,O3,O4,O5,O6,O7):
    """Paper Table 6: C-Trapezoid skewness."""
    num = 3*(O7+O1) - 2*(O6+O2) + 3*(O5+O3) - 8*O4
    den = 3*(O7-O1) - 2*(O6-O2) + 3*(O5-O3)
    return num/den if den != 0 else np.nan

def K_ctrap(O1,O2,O3,O4,O5,O6,O7):
    """C-Trapezoid kurtosis (corrected): equals (6a-6b+2c)/(2*den) with den=3a-2b+3c; matches K=(H_L+H_R)/(2H)."""
    a = O7 - O1
    b = O6 - O2
    c = O5 - O3
    den = 3*a - 2*b + 3*c
    return (3*a - 3*b + c) / den if den != 0 else np.nan

# --- MIDPOINT (crude approximation, for comparison) ---

def H_mid(O1,O2,O3,O4,O5,O6,O7):
    """Midpoint MAD = IQR/2."""
    return (O6 - O2) / 2

def G_mid(O1,O2,O3,O4,O5,O6,O7):
    """Galton skewness (midpoint approximation)."""
    den = O6 - O2
    return ((O6-O4) - (O4-O2)) / den if den != 0 else np.nan

def K_mid(O1,O2,O3,O4,O5,O6,O7):
    """Paper Table 7 midpoint row."""
    a = O7 - O1
    b = O6 - O2
    c = O5 - O3
    return (a - c) / (2*b) if b != 0 else np.nan

print('C-Trapezoid and Midpoint formulas defined.')

In [ ]:
# ==============================================
# CLASSIFIER — works for GPT2, OPT, Pythia
# ==============================================

def classify_param(name, family):
    if family == 'gpt2':
        m = re.search(r'h\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attn.c_attn.weight' in name:  return layer, 'attn_input'
        if '.attn.c_proj.weight' in name:  return layer, 'attn_output'
        if '.mlp.c_fc.weight' in name:     return layer, 'ffn_input'
        if '.mlp.c_proj.weight' in name:   return layer, 'ffn_output'
    elif family == 'opt':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.self_attn.q_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.k_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.v_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.out_proj.weight' in name: return layer, 'attn_output'
        if '.fc1.weight' in name: return layer, 'ffn_input'
        if '.fc2.weight' in name: return layer, 'ffn_output'
    elif family == 'pythia':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attention.query_key_value.weight' in name: return layer, 'attn_input'
        if '.attention.dense.weight' in name: return layer, 'attn_output'
        if '.mlp.dense_h_to_4h.weight' in name: return layer, 'ffn_input'
        if '.mlp.dense_4h_to_h.weight' in name: return layer, 'ffn_output'
    return -1, None

print('Classifier defined.')

In [ ]:
# ==============================================
# ANALYSE ONE MODEL
# ==============================================

def analyse_model(model_name, family, label):
    from transformers import AutoModelForCausalLM
    print(f'\nLoading {model_name} ...')
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()
    rows = []
    for name, p in model.named_parameters():
        layer, mtype = classify_param(name, family)
        if mtype is None: continue
        w = p.detach().cpu().numpy().astype(np.float64).ravel()
        O = compute_octiles(w)
        O1,O2,O3,O4,O5,O6,O7 = O
        rows.append({
            'model': label, 'param': name, 'layer': layer,
            'type': mtype, 'shape': str(tuple(p.shape)), 'n': w.size,
            'O1':O1,'O2':O2,'O3':O3,'O4':O4,'O5':O5,'O6':O6,'O7':O7,
            'H_ct': H_ctrap(O1,O2,O3,O4,O5,O6,O7),
            'G_ct': G_ctrap(O1,O2,O3,O4,O5,O6,O7),
            'K_ct': K_ctrap(O1,O2,O3,O4,O5,O6,O7),
            'H_mp': H_mid(O1,O2,O3,O4,O5,O6,O7),
            'G_mp': G_mid(O1,O2,O3,O4,O5,O6,O7),
            'K_mp': K_mid(O1,O2,O3,O4,O5,O6,O7),
        })
    del model; gc.collect()
    df = pd.DataFrame(rows)
    print(f'  Done: {len(df)} matrices for {label}')
    return df

print('Analysis function defined.')

In [ ]:
# ==============================================
# RUN ALL MODELS (~8 min total)
# ==============================================

MODELS = [
    ('openai-community/gpt2',        'gpt2',   'GPT2-Small'),
    ('openai-community/gpt2-medium', 'gpt2',   'GPT2-Medium'),
    ('facebook/opt-125m',            'opt',     'OPT-125M'),
    ('EleutherAI/pythia-160m',       'pythia',  'Pythia-160M'),
]

dfs = [analyse_model(n, f, l) for n, f, l in MODELS]
df = pd.concat(dfs, ignore_index=True)
df.to_csv(f'{OUT}/all_ctrap.csv', index=False)
print(f'\nTotal: {len(df)} matrices. Saved all_ctrap.csv')

In [ ]:
# ==============================================
# TABLE: Option C summary (means by model × type)
# ==============================================

summary = df.groupby(['model','type'])[['H_ct','G_ct','K_ct','H_mp','G_mp','K_mp']].mean().round(5).reset_index()
summary.to_csv(f'{OUT}/summary_ctrap.csv', index=False)
print('=== C-Trapezoid (H_ct, G_ct, K_ct) vs Midpoint (H_mp, G_mp, K_mp) ===')
display(summary)

In [ ]:
# ==============================================
# FIG 1: Histograms — GPT2-Small attn_output layers 0,4,8,11
# ==============================================

from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')

layers_to_show = [0, 4, 8, 11]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('GPT2-Small attn_output: weight distributions at different depths', fontsize=13)

for ax, lyr in zip(axes.flat, layers_to_show):
    pname = f'transformer.h.{lyr}.attn.c_proj.weight'
    w = dict(model.named_parameters())[pname].detach().cpu().numpy().ravel()
    lo, hi = np.percentile(w, [0.5, 99.5])
    ax.hist(w, bins=200, range=(lo, hi), density=True, color='steelblue', alpha=0.7, edgecolor='none')
    O = compute_octiles(w)
    for i, oi in enumerate(O):
        ax.axvline(oi, color='coral', lw=0.8, alpha=0.6)
    ax.axvline(O[3], color='red', lw=1.5)
    H = H_ctrap(*O); G = G_ctrap(*O); K = K_ctrap(*O)
    ax.set_title(f'Layer {lyr}:  H={H:.4f}  G={G:.4f}  K={K:.3f}', fontsize=10)
    ax.set_xlabel('Weight value')

del model; gc.collect()
plt.tight_layout()
plt.savefig(f'{OUT}/fig1_histograms.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig1_histograms.png')

In [ ]:
# ==============================================
# FIG 2: H vs layer depth (all 4 models)
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['H_ct'], marker='o', ms=4, label=label)
    ax.set_title(f'{ntype}: H (C-Trapezoid) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('H')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig2_H_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 3: K vs layer depth (all 4 models)
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['K_ct'], marker='o', ms=4, label=label)
    ax.set_title(f'{ntype}: K (C-Trapezoid) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('K')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig3_K_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 4: G vs layer depth (all 4 models)
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['G_ct'], marker='o', ms=4, label=label)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_title(f'{ntype}: G (C-Trapezoid) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('G')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_G_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 5: Heatmaps of H (2x2 grid, all 4 models)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('H (C-Trapezoid MAD) by layer and matrix type', fontsize=13)
for ax, (_, _, label) in zip(axes.flat, MODELS):
    sub = df[df['model']==label]
    pv = sub.pivot_table(index='type', columns='layer', values='H_ct', aggfunc='mean')
    sns.heatmap(pv, cmap='viridis', ax=ax, cbar_kws={'label':'H'})
    ax.set_title(label); ax.set_xlabel('Layer'); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_heatmap_H.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 6: Heatmaps of K (2x2 grid, all 4 models)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('K (C-Trapezoid Kurtosis) by layer and matrix type', fontsize=13)
for ax, (_, _, label) in zip(axes.flat, MODELS):
    sub = df[df['model']==label]
    pv = sub.pivot_table(index='type', columns='layer', values='K_ct', aggfunc='mean')
    sns.heatmap(pv, cmap='magma', ax=ax, cbar_kws={'label':'K'})
    ax.set_title(label); ax.set_xlabel('Layer'); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT}/fig6_heatmap_K.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 7: Boxplots of G and K by matrix type
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='type', y='K_ct', hue='model', ax=axes[0])
axes[0].set_title('K (C-Trapezoid) by matrix type'); axes[0].set_xlabel('')
sns.boxplot(data=df, x='type', y='G_ct', hue='model', ax=axes[1])
axes[1].set_title('G (C-Trapezoid) by matrix type'); axes[1].set_xlabel('')
axes[1].axhline(0, color='gray', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{OUT}/fig7_boxplots.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# PRINT SUMMARY
# ==============================================

print('='*60)
print('KEY FINDINGS (C-Trapezoid)')
print('='*60)
for label in df['model'].unique():
    sub = df[df['model']==label]
    print(f'\n--- {label} ({sub["layer"].max()+1} layers) ---')
    for nt in ['attn_output','ffn_output']:
        s = sub[sub['type']==nt].sort_values('layer')
        if len(s)>0:
            print(f'  {nt}: H [{s.H_ct.min():.4f} → {s.H_ct.max():.4f}]  K [{s.K_ct.min():.3f} → {s.K_ct.max():.3f}]')
    print(f'  G range: [{sub.G_ct.min():.4f}, {sub.G_ct.max():.4f}]')

In [ ]:
# ==============================================
# DOWNLOAD ALL
# ==============================================
try:
    from google.colab import files
    import glob
    for f in glob.glob(f'{OUT}/*'): files.download(f)
except: print('Files in:', OUT, os.listdir(OUT))